In [47]:
import json
import re
import random
from pathlib import Path
import pandas as pd

import sys, os
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.utils.json_utils import load_jsonl


DATA_DIR = Path("../data")
PROC_DIR = DATA_DIR / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH = PROC_DIR / "work_chats_dataset.jsonl"

SEED = 42
random.seed(SEED)

TARGET_GOLD = 30

### Load JSONL

In [48]:
dataset = load_jsonl(DATASET_PATH)
len(dataset), dataset[0].keys()

(177, dict_keys(['dialogue_id', 'language', 'dialogue', 'messages', 'meta']))

### Feature extraction

In [49]:
ACTION_STRONG_PAT = re.compile(
    r"\b(сдела(й|ть)|додела(й|ть)|проверь|проверить|закин(ь|уть)|пришли|отправ(ь|ить)|созвон|встреча|запили(ть|)|сверста(й|ть)|пофикси(ть|)|поправ(ь|ить))\b",
    flags=re.IGNORECASE
)

ACTION_WEAK_PAT = re.compile(
    r"\b(надо|нужно|срочно|готово|сегодня|завтра|дедлайн)\b",
    flags=re.IGNORECASE
)

QUESTION_PAT = re.compile(r"\?", flags=re.UNICODE)
URL_PAT = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)

# Сигналы "шума"/small talk: смех/сленг + последовательности скобок + некоторые эмодзи
NOISE_PAT = re.compile(
    r"(ахах|аха|хаха|лол|кек|гы+|\){2,}|\({2,}|😅|😂|🤣)",
    flags=re.IGNORECASE
)

def count_speakers(messages):
    s = set()
    for m in messages or []:
        fr = m.get("from")
        if isinstance(fr, str) and fr.strip():
            s.add(fr.strip())
    return len(s)

def extract_features(item):
    dialog = item.get("dialogue", "") or ""
    msgs = item.get("messages") or []
    n_speakers = count_speakers(msgs)
    n_msgs = len(msgs)

    n_chars = len(dialog)
    has_action_strong = bool(ACTION_STRONG_PAT.search(dialog))
    has_action_weak = bool(ACTION_WEAK_PAT.search(dialog))
    n_questions = len(QUESTION_PAT.findall(dialog))
    has_question = n_questions > 0
    has_url = bool(URL_PAT.search(dialog))
    has_noise = bool(NOISE_PAT.search(dialog))

    # доля “коротких” сообщений — приблизительный шум/реакции
    short_msgs = 0
    for m in msgs:
        t = (m.get("text") or "").strip()
        if 0 < len(t) <= 8:
            short_msgs += 1
    short_ratio = short_msgs / max(1, n_msgs)

    return {
        "dialogue_id": item.get("dialogue_id"),
        "chat_name": (item.get("meta") or {}).get("name", ""),
        "n_speakers": n_speakers,
        "n_msgs": n_msgs,
        "n_chars": n_chars,
        "has_action_strong": has_action_strong,
        "has_action_weak": has_action_weak,
        "n_questions": n_questions,
        "has_question": has_question,
        "has_url": has_url,
        "has_noise": has_noise,
        "short_ratio": short_ratio,
        # для удобного просмотра
        "preview": dialog[:220].replace("\n", " ")
    }

rows = [extract_features(x) for x in dataset]
df = pd.DataFrame(rows)
df.head()

,dialogue_id,chat_name,n_speakers,n_msgs,n_chars,has_action_strong,has_action_weak,n_questions,has_question,has_url,has_noise,short_ratio,preview
0,Chat_001__batch_00000,Chat_001,6,20,4339,False,False,1,True,False,False,0.15,2025-07-01T15:46:16 | Speaker_A: всем привет! ...
1,Chat_001__batch_00001,Chat_001,6,20,1419,False,True,5,True,False,False,0.15,2025-07-01T21:13:59 | Speaker_F: аа 2025-07-01...
2,Chat_001__batch_00002,Chat_001,4,20,12313,False,True,1,True,False,False,0.15,2025-07-02T11:49:43 | Speaker_B: Только сегодн...
3,Chat_001__batch_00003,Chat_001,6,20,1943,False,True,5,True,False,True,0.25,2025-07-02T15:03:45 | Speaker_F: Завтра: 1. Со...
4,Chat_001__batch_00004,Chat_001,7,20,4078,True,False,2,True,False,True,0.20,2025-07-02T16:27:35 | Speaker_E: 5 промтов 202...


In [50]:
print(df["has_action_strong"].mean())
print(df["has_action_weak"].mean())

0.536723163841808
0.7401129943502824


In [51]:
df["n_speakers"].describe()

count    177.000000
mean       4.225989
std        1.281336
min        1.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        7.000000
Name: n_speakers, dtype: float64

In [52]:
df["n_speakers"].value_counts().sort_index()

n_speakers
1     2
2    13
3    33
4    63
5    33
6    27
7     6
Name: count, dtype: int64

In [53]:
df["n_chars"].describe()

count      177.000000
mean      2085.344633
std       1347.395463
min         57.000000
25%       1419.000000
50%       1752.000000
75%       2300.000000
max      12313.000000
Name: n_chars, dtype: float64

In [54]:
df["n_questions"].describe()

count    177.000000
mean       3.338983
std        2.298296
min        0.000000
25%        2.000000
50%        3.000000
75%        5.000000
max       13.000000
Name: n_questions, dtype: float64

In [55]:
df["short_ratio"].describe()

count    177.000000
mean       0.157910
std        0.100253
min        0.000000
25%        0.100000
50%        0.150000
75%        0.200000
max        0.650000
Name: short_ratio, dtype: float64

In [56]:
def assign_group(r):
    # low-info: слишком коротко и почти без обсуждения
    if r["n_chars"] < 300 and r["n_speakers"] <= 2 and (not r["has_action_strong"]) and (not r["has_question"]):
        return "low_info"

    # strong action — явные поручения
    if r["has_action_strong"]:
        return "action_strong"

    # discussion — много вопросов, без явных поручений
    if r["n_questions"] >= 4 and (not r["has_action_strong"]):
        return "discussion"

    # noisy — высокий процент коротких сообщений
    if r["short_ratio"] >= 0.30:
        return "noisy"

    # всё остальное — статус/координация
    return "status"


df["group"] = df.apply(assign_group, axis=1)
df["group"].value_counts()

group
action_strong    95
status           46
discussion       29
noisy             4
low_info          3
Name: count, dtype: int64

In [57]:
spk_bins = [0, 3, 5, 7]
spk_labels = ["few", "mid", "many"]

len_bins = [0, 1400, 2300, 6000]
len_labels = ["short", "medium", "long"]

df = df.copy()
df["spk_bin"] = pd.cut(df["n_speakers"], bins=spk_bins, labels=spk_labels, include_lowest=True)
df["len_bin"] = pd.cut(df["n_chars"], bins=len_bins, labels=len_labels, include_lowest=True)

In [58]:
QUOTAS = {
    "action_strong": 15,
    "status": 8,
    "discussion": 5,
    "noisy": 1,
    "low_info": 1,
}

assert sum(QUOTAS.values()) == TARGET_GOLD

In [59]:
def stratified_sample(group_df: pd.DataFrame, k: int, seed: int = 42) -> pd.DataFrame:
    """
    Stratified sampling over (len_bin, spk_bin).
    - First, take 1 item per bucket where possible.
    - Then, fill remaining with random samples from the remaining pool.
    """
    if len(group_df) == 0 or k <= 0:
        return group_df.head(0)

    if len(group_df) <= k:
        return group_df.sample(frac=1.0, random_state=seed)

    tmp = group_df.copy()

    # Some bins may have NaN if out of range; keep them as category "NaN" buckets
    # Group by bin pairs
    buckets = []
    for (_, _), sub in tmp.groupby(["len_bin", "spk_bin"], dropna=False):
        if len(sub) == 0:
            continue
        buckets.append(sub.sample(n=1, random_state=seed))

    out = pd.concat(buckets, ignore_index=True).drop_duplicates("dialogue_id")

    # Fill remaining
    if len(out) < k:
        remaining = tmp[~tmp["dialogue_id"].isin(out["dialogue_id"])]
        need = k - len(out)
        out = pd.concat(
            [out, remaining.sample(n=min(need, len(remaining)), random_state=seed)],
            ignore_index=True
        ).drop_duplicates("dialogue_id")

    # If still >k (can happen), trim
    if len(out) > k:
        out = out.sample(n=k, random_state=seed)

    return out

In [60]:
selected_parts = []

for grp, k in QUOTAS.items():
    gdf = df[df["group"] == grp]
    if len(gdf) == 0:
        print(f"WARNING: group '{grp}' is empty; quota {k} will be reallocated later.")
        continue

    # For tiny groups, just random sample
    if len(gdf) <= max(3, k) and len(gdf) < 10:
        sel = gdf.sample(n=min(k, len(gdf)), random_state=SEED)
    else:
        sel = stratified_sample(gdf, k, seed=SEED)

    sel = sel.copy()
    sel["quota_group"] = grp
    selected_parts.append(sel)

selected_df = pd.concat(selected_parts, ignore_index=True).drop_duplicates("dialogue_id")

# If we didn't reach 30 due to empty/small groups, fill from remaining pool (prefer larger groups)
if len(selected_df) < TARGET_GOLD:
    need = TARGET_GOLD - len(selected_df)

    # Prefer remaining from action_strong -> status -> discussion, then anything
    priority_order = ["action_strong", "status", "discussion", "noisy", "low_info"]
    remaining_all = df[~df["dialogue_id"].isin(selected_df["dialogue_id"])]

    filler = []
    for grp in priority_order:
        if need <= 0:
            break
        pool = remaining_all[remaining_all["group"] == grp]
        if len(pool) == 0:
            continue
        take = min(need, len(pool))
        filler.append(pool.sample(n=take, random_state=SEED))
        need -= take
        remaining_all = remaining_all[~remaining_all["dialogue_id"].isin(filler[-1]["dialogue_id"])]

    if filler:
        selected_df = pd.concat([selected_df] + filler, ignore_index=True).drop_duplicates("dialogue_id")

# Final sanity checks
selected_df = selected_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print("Selected:", len(selected_df))
print(selected_df["group"].value_counts())

Selected: 30
group
action_strong    15
status            8
discussion        5
noisy             1
low_info          1
Name: count, dtype: int64


/var/folders/4r/kcskh83x3rj2mwg0r3tgrbxm0000gn/T/ipykernel_46869/3095974316.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (_, _), sub in tmp.groupby(["len_bin", "spk_bin"], dropna=False):
/var/folders/4r/kcskh83x3rj2mwg0r3tgrbxm0000gn/T/ipykernel_46869/3095974316.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for (_, _), sub in tmp.groupby(["len_bin", "spk_bin"], dropna=False):
/var/folders/4r/kcskh83x3rj2mwg0r3tgrbxm0000gn/T/ipykernel_46869/3095974316.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=Fal

In [61]:
gold_ids = selected_df["dialogue_id"].tolist()

gold_ids_path = PROC_DIR / "gold_subset_ids.json"
with gold_ids_path.open("w", encoding="utf-8") as f:
    json.dump(gold_ids, f, ensure_ascii=False, indent=2)

preview_path = PROC_DIR / "gold_subset_preview.csv"
cols = ["dialogue_id", "chat_name", "group", "n_speakers", "spk_bin", "n_chars", "len_bin", "n_questions", "short_ratio", "preview"]
selected_df[cols].to_csv(preview_path, index=False, encoding="utf-8")

gold_ids_path, preview_path, gold_ids[:5]

(PosixPath('../data/processed/gold_subset_ids.json'),
 PosixPath('../data/processed/gold_subset_preview.csv'),
 ['Chat_001__batch_00014',
  'Chat_001__batch_00002',
  'Chat_001__batch_00067',
  'Chat_002__batch_00023',
  'Chat_001__batch_00065'])